# 멀티시드 풀파이프라인 배깅 (10개 시드 평균) + 최종 제출 파일 생성

지금까지 시도한 것들은 전부 "더 많은 신호를 찾기"였는데, 이번엔 성격이
다릅니다: **이미 가진 신호를, 운에 덜 좌우되게 안정적으로 뽑아내기**입니다.

`StratifiedKFold`와 `LGBMClassifier`의 `random_state`를 10가지 다른 값으로
바꿔서 전체 파이프라인(5-Fold 학습+예측)을 10번 반복하고, **test 예측을
10번 평균**냅니다. 단일 시드(1004) 하나로 운이 좋거나 나쁜 결과 대신,
좀 더 "중심값"에 가까운 예측을 만드는 게 목표입니다.

torch 없이 LightGBM만 사용합니다.

**예상 시간**: 단일 시드 5-Fold가 몇 분 걸렸다면, 10배인 셈이라 20~40분 예상.

## 1. 라이브러리, 설정, best_params 로드

In [1]:
import os
import json
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
CACHE_DIR = "blend_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SEEDS = [1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]
print(f"사용할 시드 {len(SEEDS)}개: {SEEDS}")

with open("best_params.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
best_params = loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded
print("best_params 키:", list(best_params.keys()))

사용할 시드 10개: [1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]
best_params 키: ['n_estimators', 'learning_rate', 'num_leaves', 'max_depth', 'min_child_samples', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'min_split_gain']


## 2. 데이터 전처리 (train/test 동시, 안전한 라벨인코딩)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])
test_raw = pd.read_csv(TEST_PATH)
test_ids = test_raw["ID"].copy()
test_raw = test_raw.drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
df_test = fill_na(base_features(test_raw.copy()).drop(columns=DEAD_COLS))

cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

# train으로만 LabelEncoder를 fit하고, test에만 있는 새 라벨은 classes_에 추가
# (main.py와 동일한 안전한 방식, leakage 없음)
for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col].astype(str))

    test_vals = df_test[col].astype(str)
    unseen = sorted(set(test_vals.unique()) - set(le.classes_))
    if unseen:
        le.classes_ = np.concatenate([le.classes_, unseen])
    df_test[col] = le.transform(test_vals)

y = df_train[TARGET_COL].values.astype(np.float32)
X_train = df_train.drop(columns=[TARGET_COL])
X_test = df_test[X_train.columns]  # 컬럼 순서 일치

print(f"전처리 완료: train {X_train.shape}, test {X_test.shape}")

전처리 완료: train (256351, 67), test (90067, 67)


## 3. 멀티시드 5-Fold 반복 (OOF + test 예측)

In [3]:
oof_per_seed = []
test_pred_per_seed = []

for seed_i, seed in enumerate(SEEDS, start=1):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(y))
    test_pred_seed = np.zeros(len(X_test))

    for tr_idx, val_idx in skf.split(X_train, y):
        m = LGBMClassifier(**best_params, random_state=seed, verbose=-1)
        m.fit(X_train.iloc[tr_idx], y[tr_idx])
        oof_seed[val_idx] = m.predict_proba(X_train.iloc[val_idx])[:, 1]
        test_pred_seed += m.predict_proba(X_test)[:, 1] / N_SPLITS

    seed_auc = roc_auc_score(y, oof_seed)
    oof_per_seed.append(oof_seed)
    test_pred_per_seed.append(test_pred_seed)
    print(f"시드 {seed_i}/{len(SEEDS)} (random_state={seed})  OOF AUC: {seed_auc:.5f}")

oof_per_seed = np.array(oof_per_seed)       # shape: (n_seeds, n_train)
test_pred_per_seed = np.array(test_pred_per_seed)  # shape: (n_seeds, n_test)

시드 1/10 (random_state=1004)  OOF AUC: 0.73998
시드 2/10 (random_state=42)  OOF AUC: 0.73998
시드 3/10 (random_state=7)  OOF AUC: 0.73997
시드 4/10 (random_state=123)  OOF AUC: 0.73997
시드 5/10 (random_state=2024)  OOF AUC: 0.74006
시드 6/10 (random_state=88)  OOF AUC: 0.74009
시드 7/10 (random_state=555)  OOF AUC: 0.73975
시드 8/10 (random_state=999)  OOF AUC: 0.73993
시드 9/10 (random_state=31)  OOF AUC: 0.74019
시드 10/10 (random_state=77)  OOF AUC: 0.74008


## 4. 시드별 변동성 확인 + 평균 OOF 점수

In [4]:
individual_scores = np.array([roc_auc_score(y, oof_per_seed[i]) for i in range(len(SEEDS))])
print(f"시드별 OOF AUC: {[f'{s:.5f}' for s in individual_scores]}")
print(f"평균: {individual_scores.mean():.5f}  |  표준편차: {individual_scores.std():.5f}")
print(f"최소: {individual_scores.min():.5f}  |  최대: {individual_scores.max():.5f}")
print()

oof_bagged = oof_per_seed.mean(axis=0)
score_bagged = roc_auc_score(y, oof_bagged)
print(f"10개 시드 평균낸 OOF의 전체 AUC: {score_bagged:.5f}")
print(f"단일 시드(1004) 기준 AUC(0.73998) 대비: {score_bagged - 0.73998:+.5f}")
print()
print("참고: 시드별 표준편차가 이게 바로 \"노이즈 크기\"의 실측값입니다.")
print("우리와 1위의 차이(0.00075)가 이 표준편차보다 작다면, 그 차이는 순전히 운일 가능성이 높습니다.")

시드별 OOF AUC: ['0.73998', '0.73998', '0.73997', '0.73997', '0.74006', '0.74009', '0.73975', '0.73993', '0.74019', '0.74008']
평균: 0.74000  |  표준편차: 0.00011
최소: 0.73975  |  최대: 0.74019

10개 시드 평균낸 OOF의 전체 AUC: 0.74036
단일 시드(1004) 기준 AUC(0.73998) 대비: +0.00038

참고: 시드별 표준편차가 이게 바로 "노이즈 크기"의 실측값입니다.
우리와 1위의 차이(0.00075)가 이 표준편차보다 작다면, 그 차이는 순전히 운일 가능성이 높습니다.


## 5. 최종 제출 파일 생성 (10개 시드 평균 test 예측)

In [6]:
test_pred_bagged = test_pred_per_seed.mean(axis=0)

submission = pd.DataFrame({
    "ID": test_ids,
    "probability": test_pred_bagged,
})

output_path = "1차_LGBM_10seed_bagged_submit.csv"
submission.to_csv(output_path, index=False)
print(f"저장 완료: {output_path}")
print(submission.head())

np.save(f"{CACHE_DIR}/test_lgbm_bagged.npy", test_pred_bagged)

# OOF도 보관 (참고용)
np.save(f"{CACHE_DIR}/oof_10seed_bagged.npy", oof_bagged)
print("\n참고용 저장: blend_cache/oof_10seed_bagged.npy")

저장 완료: 1차_LGBM_10seed_bagged_submit.csv
           ID  probability
0  TEST_00000     0.001374
1  TEST_00001     0.002161
2  TEST_00002     0.152768
3  TEST_00003     0.107314
4  TEST_00004     0.512295

참고용 저장: blend_cache/oof_10seed_bagged.npy


## 6. 다음 단계

이 CSV가 바로 **"운에 덜 좌우되는" 후보 제출 파일**입니다. 단일 시드보다
로컬 OOF 점수가 크게 다르진 않겠지만(노이즈 자체를 없애는 게 아니라
평균내는 거라), 실제 리더보드에서는 단일 시드의 "재수 없는 한 번"이 아니라
좀 더 안정적인 결과를 기대할 수 있습니다.

팀원들과 상의해서, 이 파일을 제출 후보로 검토해보세요.